# kinfast on a GPU

Run this on a Colab **T4 GPU** runtime (Runtime > Change runtime type > T4 GPU).
It does three things, in order:

1. checks that every module gives the same answers on CUDA as on CPU
2. measures batched FK, Jacobian and IK throughput and writes `BENCHMARK_GPU.md`
3. renders the 10,000 arm demo gif

Run every cell top to bottom, then copy the two result blocks at the end back
into the repo. Total time is about ten minutes, most of it the pip install.

## 1. Setup

If the GPU check below prints `no CUDA device`, the runtime is still on CPU:
switch it in Runtime > Change runtime type and run the cell again.

In [ ]:
import torch
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no CUDA device")

In [ ]:
import os
if not os.path.isdir("kinfast"):
    !git clone -q https://github.com/VihanAggarwal/kinfast
%cd kinfast
# torch is already installed on Colab with CUDA, so do not let pip replace it
!pip install -q -e . --no-deps
!pip install -q pytest mujoco pytorch-kinematics xacro hypothesis
import kinfast
print("kinfast", kinfast.__version__)

In [ ]:
# the robot files are fetched, not vendored, so grab them
!python examples/gallery.py --fetch 2>&1 | tail -3
!ls examples/assets/gallery | head

## 2. Correctness first

Every module has to agree with the CPU path before any number is worth
measuring. These tests skip themselves without CUDA, so on a GPU runtime they
should all run and pass.

In [ ]:
!python -m pytest tests/test_gpu.py -v

## 3. The benchmark

Batches grow by powers of ten until the card runs out of memory, with explicit
synchronization around every timed call so a kernel launch is never mistaken
for finished work. The result lands in `examples/assets/BENCHMARK_GPU.md`.

In [ ]:
!python examples/gpu_benchmark.py

## 4. The 10,000 arm demo

This is the gif for the README: ten thousand IK problems solved in one batch.

In [ ]:
!python examples/demo_10k_arms.py --urdf examples/assets/gallery/panda.urdf \
    --n 10000 --restarts 4 --gif demo_gpu.gif
from IPython.display import Image, display
display(Image("demo_gpu.gif"))

## 5. What to bring home

Two artifacts: the benchmark table below, and `demo_gpu.gif` from the file
browser on the left (right click > Download). Paste the table into the
README's speed section and commit the gif.

In [ ]:
print(open("examples/assets/BENCHMARK_GPU.md").read())

In [ ]:
# a one line summary worth keeping next to the table
import time, torch, kinfast
robot = kinfast.load("examples/assets/gallery/panda.urdf").to("cuda")
q = robot.random_configs(100_000)
robot.fk_all(q); torch.cuda.synchronize()
t0 = time.perf_counter(); robot.fk_all(q); torch.cuda.synchronize()
dt = time.perf_counter() - t0
print(f"{torch.cuda.get_device_name(0)}: forward kinematics for 100,000 "
      f"configurations in {dt*1e3:.1f} ms ({100_000/dt:,.0f} per second)")